In [1]:
from collections import deque
import random
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

ModuleNotFoundError: No module named 'gymnasium'

In [ ]:
# Separate function/class to build the Q-Network architecture
def build_q_network(state_dim=2, action_dim=3, hidden_dim=64):
  """Builds a Deep Q-Network function approximator."""
  return nn.Sequential(
      nn.Linear(state_dim, hidden_dim),
      nn.ReLU(),
      nn.Linear(hidden_dim, hidden_dim),
      nn.ReLU(),
      nn.Linear(hidden_dim, action_dim),
  )

In [2]:
# Experience Replay Buffer using python collections.deque
class ReplayBuffer:

  def __init__(self, capacity=10000):
    self.buffer = deque(maxlen=capacity)

  def push(self, state, action, reward, next_state, done):
    self.buffer.append((state, action, reward, next_state, done))

  def sample(self, batch_size):
    batch = random.sample(self.buffer, batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)
    return (
        torch.tensor(states, dtype=torch.float32),
        torch.tensor(actions, dtype=torch.int64),
        torch.tensor(rewards, dtype=torch.float32),
        torch.tensor(next_states, dtype=torch.float32),
        torch.tensor(dones, dtype=torch.float32),
    )

  def __len__(self):
    return len(self.buffer)

In [3]:
# Environment &amp; Hyperparameters
env = gym.make("MountainCar-v0")
state_dim = env.observation_space.shape  # 2 (position, velocity)
action_dim = env.action_space.n  # 3 discrete actions

gamma = 0.99  # Discount factor
lr = 0.001  # Learning rate
epsilon = 1.0  # Initial exploration rate
epsilon_min = 0.01  # Minimum exploration rate
epsilon_decay = 0.995  # Decay per episode
batch_size = 64  # Mini-batch size
target_update_freq = 10  # Copy target weights every N episodes

# Instantiate Networks, Memory Buffer, &amp; Optimizer
q_net = build_q_network(state_dim, action_dim)
target_net = build_q_network(state_dim, action_dim)
target_net.load_state_dict(q_net.state_dict())  # Synchronize initial weights

optimizer = optim.Adam(q_net.parameters(), lr=lr)
loss_fn = nn.MSELoss()
replay_buffer = ReplayBuffer(capacity=20000)

num_episodes = 800
goal_reached_count = 0

NameError: name 'gym' is not defined

In [ ]:
# MAIN TRAINING LOOP
for episode in range(1, num_episodes + 1):
  state, info = env.reset()  # Reset environment at episode start
  total_reward = 0
  done = False

  while not done:
    # 1. Epsilon-Greedy Action Selection
    if random.random() &lt; epsilon:
      action = env.action_space.sample()  # Explore randomly
    else:
      state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
      with torch.no_grad():
        q_values = q_net(state_tensor)
        action = q_values.argmax().item()  # Exploit highest predicted Q-value

    # 2. Transition Step in Environment
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    # Optional: Custom reward modification to encourage reaching higher position
    # (MountainCar position ranges from -1.2 to 0.6; goal is at position &gt;= 0.5)
    # Apply Custom Reward Shaping
    pos, vel = next_state, next_state[3] 
    shaped_reward = reward + (pos + 0.5) + (abs(vel) * 10.0)

    # 3. Store Experience Tuple in Replay Buffer
    replay_buffer.push(state, action, shaped_reward, next_state, done)
    state = next_state
    total_reward += reward

    # 4. Train Network on Mini-batch from Replay Memory
    if len(replay_buffer) &gt;= batch_size:
      b_states, b_actions, b_rewards, b_next_states, b_dones = (
          replay_buffer.sample(batch_size)
      )

      # Current predicted Q-values: Q(s, a)
      q_eval = q_net(b_states).gather(1, b_actions.unsqueeze(1)).squeeze(1)

      # Bellman Target Calculation using Target Network: Q_target = R + gamma * max(Q(s', a'))
      with torch.no_grad():
        max_next_q = target_net(b_next_states).max(1)
        q_target = b_rewards + (gamma * max_next_q * (1 - b_dones))

      # Compute MSE Loss and perform Gradient Descent
      loss = loss_fn(q_eval, q_target)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # Decay Epsilon after each episode
  epsilon = max(epsilon_min, epsilon * epsilon_decay)

  # Periodically update Target Network weights
  if episode % target_update_freq == 0:
    target_net.load_state_dict(q_net.state_dict())

  # Log progress and check if goal flag was reached
  if next_state &gt;= 0.5:
    goal_reached_count += 1
    print(
        f"Episode {episode} | GOAL REACHED! Final Position: {next_state:.2f}"
    )
  elif episode % 20 == 0:
    print(
        f"Episode {episode}/{num_episodes} | Total Reward: {total_reward} |"
        f" Epsilon: {epsilon:.3f}"
    )

  # Save model state upon successful goal achievement threshold
  if goal_reached_count &gt;= 5:
    torch.save(q_net.state_dict(), "dqn_mountaincar.pth")
    print("\nModel saved successfully as 'dqn_mountaincar.pth'!")
    break

env.close()

In [ ]:
# EVALUATION LOOP
test_env = gym.make("MountainCar-v0", render_mode="human")
policy_net = build_q_network(state_dim, action_dim)
policy_net.load_state_dict(torch.load("dqn_mountaincar.pth"))
policy_net.eval()

print("\n--- Running 5 Flawless Test Episodes ---")
for test_ep in range(1, 6):
  state, info = test_env.reset()
  done = False
  steps = 0

  while not done:
    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
      action = policy_net(state_tensor).argmax().item()

    state, reward, terminated, truncated, info = test_env.step(action)
    done = terminated or truncated
    steps += 1

  print(
      f"Test Episode {test_ep}: Reached Flag in {steps} steps! Final Position:"
      f" {state:.2f}"
  )

test_env.close()